# Proyecto final: dataset real 4x4 (Sector Minería de México) -> QUBO -> QAOA local

Qubit.mx, QMexico Summer School 2026.

Este notebook toma datos reales del sector Minería de México (portal DataMéxico, Secretaría de Economía), los convierte en un problema de matching bipartito 4x4, lo formula como QUBO, lo valida con un método clásico exacto y lo resuelve con QAOA local.

Fuente de los datos:
https://www.economia.gob.mx/datamexico/es/profile/industry/mining-quarrying-and-oil-and-gas-extraction

A diferencia del ejemplo molecular del notebook base, aquí el dataset es real y está documentado en el README del repositorio.

## 0. Instalación del entorno

Ejecuta esta celda al iniciar la sesión en Google Colab. Instala las librerías científicas y Qiskit.

In [ ]:
%%capture
%pip -q install numpy pandas scipy matplotlib qiskit qiskit-aer pylatexenc

## 1. Idea del proyecto en una línea

El flujo de razonamiento es el siguiente:

```text
datos reales del sector Minería -> dos conjuntos A y B -> score S_ij -> variables binarias x_ij -> restricciones -> QUBO -> solución
```

La pregunta central no es solo qué dataset usamos, sino por qué se puede modelar de forma honesta como un matching bipartito 4x4.

## 2. Definición de los dos lados A y B

Elegimos como conjuntos:

A = cuatro años recientes del sector Minería: 2021, 2022, 2023, 2024.

B = cuatro tamaños de empresa: Micro (hasta 10 personas), Pequeña (11 a 50), Mediana (51 a 250) y Grande (251 y más).

La variable binaria es:

x_ij = 1 si el año i se empareja con el tamaño de empresa j, y 0 en otro caso.

La interpretación es un ejercicio educativo: buscamos la asignación uno a uno año-tamaño que maximiza un score combinado de salud económica del sector. No es una recomendación de política pública.

## 3. Construcción del score S_ij

Usamos la fórmula pedida:

S_ij = 0.30 (PIB) + 0.25 (Empleo) + 0.20 (Salario) + 0.15 (Empresas) + 0.10 (Exportaciones)

Cada componente se explica así:

PIB: producto interno bruto anual del sector Minería, depende del año i.

Empleo: fuerza laboral promedio del sector en el año i.

Salario: salario mensual promedio del sector en el año i.

Empresas: porcentaje de permanencia del personal según el tamaño de empresa j (estabilidad laboral).

Exportaciones: contexto de inversión extranjera directa (IED) del sector en 2024. Es un solo valor global del sector, así que entra como un factor constante normalizado a 1.0 para todos los pares.

Cada componente se normaliza al rango [0, 1] antes de combinarse, para que los pesos sean comparables.

In [ ]:
import numpy as np
import pandas as pd

# Datos reales agregados desde los JSON de DataMexico (sector Mineria)
# PIB anual del sector (pesos)
PIB_anual = {2021: 1005807500000, 2022: 1037060000000, 2023: 1054839000000, 2024: 951349750000}

# Salario mensual promedio del sector (promedio de los 4 trimestres)
SAL_anual = {2021: 8676.33, 2022: 9394.39, 2023: 9942.16, 2024: 10248.94}

# Empleo promedio del sector (fuerza laboral, promedio de los 4 trimestres)
EMP_anual = {2021: 227026.5, 2022: 296375.0, 2023: 282816.0, 2024: 298017.5}

# Permanencia del personal por tamano de empresa (porcentaje)
PERM_tamano = {"Micro": 89.43, "Pequena": 87.11, "Mediana": 79.58, "Grande": 83.75}

# IED positiva total del sector en 2024 (contexto, factor constante)
IED_total_pos = 2936800346.37

anios = [2021, 2022, 2023, 2024]
tamanos = ["Micro", "Pequena", "Mediana", "Grande"]
print("A (anios):", anios)
print("B (tamanos):", tamanos)

A (anios): [2021, 2022, 2023, 2024]
B (tamanos): ['Micro', 'Pequena', 'Mediana', 'Grande']


In [ ]:
def normalizar(d):
    vals = np.array(list(d.values()), dtype=float)
    lo, hi = vals.min(), vals.max()
    return {k: (v - lo) / (hi - lo) for k, v in d.items()}

PIB = normalizar(PIB_anual)
SAL = normalizar(SAL_anual)
EMP = normalizar(EMP_anual)
PERM = normalizar(PERM_tamano)
EXPORT = 1.0  # factor de exportaciones/IED, constante para todos los pares

S = np.zeros((4, 4))
for i, a in enumerate(anios):
    for j, b in enumerate(tamanos):
        S[i, j] = (0.30 * PIB[a] + 0.25 * EMP[a] + 0.20 * SAL[a]
                   + 0.15 * PERM[b] + 0.10 * EXPORT)

S_df = pd.DataFrame(S, index=anios, columns=tamanos)
print("Matriz de score S_ij:")
S_df.round(4)

Matriz de score S_ij:


,Micro,Pequena,Mediana,Grande
2021,0.4079,0.3725,0.2579,0.3214
2022,0.8340,0.7987,0.6840,0.7475
2023,0.9075,0.8721,0.7575,0.8210
2024,0.7000,0.6647,0.5500,0.6135


## 4. Guardar el dataset como CSV

Guardamos la matriz en formato largo en `data/dataset_real_4x4.csv`. Este archivo es el que vive en el repositorio.

In [ ]:
filas = []
for i, a in enumerate(anios):
    for j, b in enumerate(tamanos):
        filas.append({"A_anio": a, "B_tamano": b, "score": round(S[i, j], 6)})
dataset_df = pd.DataFrame(filas)
dataset_df.to_csv("dataset_real_4x4.csv", index=False) #data/dataset_real_4x4.csv
print("CSV guardado. Primeras filas:")
dataset_df.head()

CSV guardado. Primeras filas:


,A_anio,B_tamano,score
0,2021,Micro,0.407865
1,2021,Pequena,0.372535
2,2021,Mediana,0.257865
3,2021,Grande,0.321367
4,2022,Micro,0.833998


## 5. Objetivo y restricciones del matching

Queremos maximizar el score total de la asignación uno a uno:

max sobre x de la suma de S_ij * x_ij

Restricción por filas (cada año se asigna exactamente a un tamaño):

la suma sobre j de x_ij = 1 para cada i

Restricción por columnas (cada tamaño recibe exactamente un año):

la suma sobre i de x_ij = 1 para cada j

Esto es exactamente un matching bipartito perfecto entre A y B.

## 6. Formulación QUBO

Un QUBO es una minimización cuadrática binaria. Convertimos el problema en:

E(x) = - suma de S_ij x_ij + lamA * suma_i (suma_j x_ij - 1)^2 + lamB * suma_j (suma_i x_ij - 1)^2

El primer término premia scores altos (con signo negativo, porque minimizamos). Los términos con lamA y lamB penalizan violar las restricciones de filas y columnas.

Si las penalizaciones son suficientemente grandes comparadas con el rango del score, el mínimo del QUBO coincide con una asignación factible de score máximo. Como el score está en [0, 1], usar lamA = lamB = 5.0 es más que suficiente.

In [ ]:
lamA = 5.0
lamB = 5.0

def indice(i, j):
    return i * 4 + j  # mapea el par (i, j) a una variable de 0 a 15

N = 16  # 4x4 variables binarias

def energia(x_plano):
    x = np.array(x_plano).reshape(4, 4)
    e = -(S * x).sum()
    for i in range(4):
        e += lamA * (x[i, :].sum() - 1) ** 2
    for j in range(4):
        e += lamB * (x[:, j].sum() - 1) ** 2
    return e

# Construccion de la matriz Q (forma x^T Q x + offset)
Q = np.zeros((N, N))
offset = 0.0
for i in range(4):
    for j in range(4):
        Q[indice(i, j), indice(i, j)] += -S[i, j]
for i in range(4):
    vs = [indice(i, j) for j in range(4)]
    for a in vs:
        Q[a, a] += lamA * (1 - 2)
    for p in range(len(vs)):
        for q in range(p + 1, len(vs)):
            Q[vs[p], vs[q]] += 2 * lamA
    offset += lamA
for j in range(4):
    vs = [indice(i, j) for i in range(4)]
    for a in vs:
        Q[a, a] += lamB * (1 - 2)
    for p in range(len(vs)):
        for q in range(p + 1, len(vs)):
            Q[vs[p], vs[q]] += 2 * lamB
    offset += lamB

print("Matriz Q construida de tamano", Q.shape)

Matriz Q construida de tamano (16, 16)


# Parte B: Validación clásica exacta

Como el problema es pequeño (16 variables binarias, 2^16 = 65536 combinaciones), podemos recorrer todas las posibilidades y encontrar el mínimo exacto. Esto nos da la respuesta de referencia contra la cual comparar QAOA.

In [6]:
import itertools

mejor_energia = None
mejor_x = None
for bits in itertools.product([0, 1], repeat=16):
    x = np.array(bits)
    e = energia(x)
    if mejor_energia is None or e < mejor_energia:
        mejor_energia = e
        mejor_x = x.copy()

best_energy_exact = mejor_energia
best_x_exact = mejor_x.reshape(4, 4)
best_score_exact = (S * best_x_exact).sum()

print("Energia minima clasica:", round(best_energy_exact, 4))
print("Score de la mejor asignacion:", round(best_score_exact, 4))
print("Asignacion (filas=anios, columnas=tamanos):")
print(pd.DataFrame(best_x_exact, index=anios, columns=tamanos))

Energia minima clasica: -2.5775
Score de la mejor asignacion: 2.5775
Asignacion (filas=anios, columnas=tamanos):
      Micro  Pequena  Mediana  Grande
2021      0        0        0       1
2022      0        0        1       0
2023      1        0        0       0
2024      0        1        0       0


In [7]:
def pares_seleccionados(x_matriz):
    pares = []
    for i in range(4):
        for j in range(4):
            if x_matriz[i, j] == 1:
                pares.append({"anio": anios[i], "tamano": tamanos[j],
                              "score": round(S[i, j], 4)})
    return pd.DataFrame(pares)

factible = all(best_x_exact.sum(1) == 1) and all(best_x_exact.sum(0) == 1)
print("La solucion es factible:", factible)
pares_seleccionados(best_x_exact)

La solucion es factible: True


,anio,tamano,score
0,2021,Grande,0.3214
1,2022,Mediana,0.6840
2,2023,Micro,0.9075
3,2024,Pequena,0.6647


# Parte C: QAOA local

Ahora resolvemos el mismo QUBO con QAOA (Quantum Approximate Optimization Algorithm) en un simulador local. Convertimos el QUBO a un Hamiltoniano de Ising, construimos el circuito QAOA con profundidad p = 1, optimizamos los parámetros con un optimizador clásico y medimos.

In [8]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit_aer import AerSimulator
from scipy.optimize import minimize

# Conversion QUBO -> Ising: x_i = (1 - z_i) / 2
h = np.zeros(N)
J = np.zeros((N, N))
const = offset
for i in range(N):
    const += Q[i, i] * 0.5
    h[i] += -0.5 * Q[i, i]
    for j in range(i + 1, N):
        const += 0.25 * Q[i, j]
        h[i] += -0.25 * Q[i, j]
        h[j] += -0.25 * Q[i, j]
        J[i, j] += 0.25 * Q[i, j]

def energia_bits(cadena_bits):
    x = np.array([int(c) for c in cadena_bits[::-1]])
    return x @ Q @ x + offset

simulador = AerSimulator()

In [9]:
def circuito_qaoa(gamma, beta):
    qc = QuantumCircuit(N)
    qc.h(range(N))  # superposicion inicial
    # capa de costo
    for i in range(N):
        if abs(h[i]) > 1e-9:
            qc.rz(2 * gamma * h[i], i)
    for i in range(N):
        for j in range(i + 1, N):
            if abs(J[i, j]) > 1e-9:
                qc.cx(i, j)
                qc.rz(2 * gamma * J[i, j], j)
                qc.cx(i, j)
    # capa mezcladora
    for i in range(N):
        qc.rx(2 * beta, i)
    qc.measure_all()
    return qc

def energia_esperada(params):
    gamma, beta = params
    qc = circuito_qaoa(gamma, beta)
    conteos = simulador.run(qc, shots=2048).result().get_counts()
    total = sum(conteos.values())
    e = 0.0
    for bs, c in conteos.items():
        e += energia_bits(bs) * c / total
    return e

resultado = minimize(energia_esperada, [0.5, 0.5], method="COBYLA",
                     options={"maxiter": 60})
gamma_opt, beta_opt = resultado.x
best_expected_energy = resultado.fun
print("Parametros optimos: gamma =", round(gamma_opt, 4), "beta =", round(beta_opt, 4))
print("Energia esperada minima:", round(best_expected_energy, 4))

Parametros optimos: gamma = 2.5182 beta = 0.3381
Energia esperada minima: 25.5558


In [10]:
qc_final = circuito_qaoa(gamma_opt, beta_opt)
counts = simulador.run(qc_final, shots=4096).result().get_counts()

mejor_bits = min(counts, key=lambda bs: energia_bits(bs))
best_observed_energy = energia_bits(mejor_bits)
x_qaoa = np.array([int(c) for c in mejor_bits[::-1]]).reshape(4, 4)

print("Mejor energia observada por QAOA:", round(best_observed_energy, 4))
print("Asignacion encontrada por QAOA:")
print(pd.DataFrame(x_qaoa, index=anios, columns=tamanos))
factible_qaoa = all(x_qaoa.sum(1) == 1) and all(x_qaoa.sum(0) == 1)
print("Es factible:", factible_qaoa)

Mejor energia observada por QAOA: -2.5775
Asignacion encontrada por QAOA:
      Micro  Pequena  Mediana  Grande
2021      0        1        0       0
2022      1        0        0       0
2023      0        0        0       1
2024      0        0        1       0
Es factible: True


In [11]:
total = sum(counts.values())
prob_feasible = 0
prob_exact_optimum = 0
for bs, c in counts.items():
    xb = np.array([int(z) for z in bs[::-1]]).reshape(4, 4)
    if all(xb.sum(1) == 1) and all(xb.sum(0) == 1):
        prob_feasible += c
    if abs(energia_bits(bs) - best_energy_exact) < 1e-3:
        prob_exact_optimum += c
prob_feasible /= total
prob_exact_optimum /= total
print("Probabilidad de medir una solucion factible:", round(prob_feasible, 4))
print("Probabilidad de medir el optimo clasico:", round(prob_exact_optimum, 4))

Probabilidad de medir una solucion factible: 0.0039
Probabilidad de medir el optimo clasico: 0.0039


# Parte D: Comparación y resumen

Comparamos el método clásico exacto contra QAOA local en la misma instancia.

In [12]:
comparacion = pd.DataFrame([
    {"metodo": "Clasico exacto", "energia": round(best_energy_exact, 4),
     "score": round(best_score_exact, 4), "factible": True},
    {"metodo": "QAOA local", "energia": round(best_observed_energy, 4),
     "score": round((S * x_qaoa).sum(), 4), "factible": bool(factible_qaoa)},
])
print(comparacion.to_string(index=False))

        metodo  energia  score  factible
Clasico exacto  -2.5775 2.5775      True
    QAOA local  -2.5775 2.5775      True


In [13]:
print("Resumen automatico")
print("------------------")
print("Instancia: Sector Mineria de Mexico, anios 2021-2024 vs tamanos de empresa")
print("Dataset:", f"{len(anios)} x {len(tamanos)}")
print("Mejor energia clasica:", round(best_energy_exact, 4))
print("Mejor score clasico:", round(best_score_exact, 4))
print()
print("Mejor asignacion clasica:")
print(pares_seleccionados(best_x_exact).to_string(index=False))
print()
print("QAOA local")
print("----------")
print("Energia esperada:", round(best_expected_energy, 4))
print("Mejor energia observada:", round(best_observed_energy, 4))
print("Probabilidad de factibilidad:", round(prob_feasible, 4))
print("Probabilidad del optimo clasico:", round(prob_exact_optimum, 4))

Resumen automatico
------------------
Instancia: Sector Mineria de Mexico, anios 2021-2024 vs tamanos de empresa
Dataset: 4 x 4
Mejor energia clasica: -2.5775
Mejor score clasico: 2.5775

Mejor asignacion clasica:
 anio  tamano  score
 2021  Grande 0.3214
 2022 Mediana 0.6840
 2023   Micro 0.9075
 2024 Pequena 0.6647

QAOA local
----------
Energia esperada: 25.5558
Mejor energia observada: -2.5775
Probabilidad de factibilidad: 0.0039
Probabilidad del optimo clasico: 0.0039


## Interpretación de los resultados

1. La mejor asignación encontrada empareja cada año con un tamaño de empresa de forma uno a uno, maximizando el score combinado de salud económica del sector Minería.

2. El método clásico exacto y QAOA local coinciden en el óptimo en esta instancia pequeña. Contra un método exacto, lo mejor que puede hacer QAOA es empatar.

3. La asignación cumple todas las restricciones de matching bipartito (cada fila y cada columna suman exactamente 1).

4. QAOA local sí logra observar el óptimo clásico entre sus mediciones, aunque con baja probabilidad porque p = 1 es la profundidad mínima.

## Limitaciones

El modelo 4x4 es un ejercicio educativo. El score mezcla variables de naturaleza distinta (PIB, empleo, salario, permanencia, IED) mediante una normalización simple y pesos fijos, lo que es una simplificación fuerte. La componente de exportaciones entra como constante porque solo existe un valor global del sector, así que en la práctica no diferencia entre pares.

Si el dataset creciera (más años o más categorías), el número de variables crecería como el producto de ambos lados y la validación exacta dejaría de ser viable, momento en el que QAOA u otros métodos heurísticos cobran más sentido.

## Advertencia ética

Los resultados de QAOA no deben presentarse como una recomendación real de política pública, inversión o asignación de recursos. Los datos provienen de fuentes públicas agregadas de DataMéxico y no contienen información personal identificable.